# C-Index STEP=2 거래비용 평가 전용 노트북

이 노트북은 모델 학습, XAI 추출, C-index 재계산을 하지 않는다.  
이미 생성된 pkl 캐시 파일을 불러와서 거래비용 반영 성과만 계산한다.

필요한 캐시 파일:

```text
/content/drive/MyDrive/c-index/artifacts/finalDf_cache.pkl
/content/drive/MyDrive/c-index/artifacts/oep_signals_cache.pkl
/content/drive/MyDrive/c-index/artifacts/analysisDf_cache.pkl
```



In [ ]:
# Colab Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRADING_DAYS_PER_YEAR = 252
SHARPE_EPSILON = 1e-6
MIN_TRADES_POOLED = 15

CACHE_DIR = Path('/content/drive/MyDrive/c-index/artifacts')
FINAL_DF_CACHE = CACHE_DIR / 'finalDf_cache.pkl'
OEP_SIGNALS_CACHE = CACHE_DIR / 'oep_signals_cache.pkl'
ANALYSIS_DF_CACHE = CACHE_DIR / 'analysisDf_cache.pkl'

RESULTS_DIR = Path('/content/drive/MyDrive/c-index/results')
OUTPUT_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if not FINAL_DF_CACHE.exists():
    raise FileNotFoundError(f'finalDf cache not found: {FINAL_DF_CACHE}')

finalDf = pd.read_pickle(FINAL_DF_CACHE)
finalDf['date'] = pd.to_datetime(finalDf['date'])
finalDf = finalDf.sort_values(['model', 'asset', 'date']).reset_index(drop=True)

# Optional caches, useful for inspection only.
oep_signals = pd.read_pickle(OEP_SIGNALS_CACHE) if OEP_SIGNALS_CACHE.exists() else None
analysisDf = pd.read_pickle(ANALYSIS_DF_CACHE) if ANALYSIS_DF_CACHE.exists() else None

print(f'[cache loaded] finalDf: {FINAL_DF_CACHE} | shape={finalDf.shape}')
print('models:', sorted(finalDf['model'].unique()))
print('assets:', sorted(finalDf['asset'].unique()))
print('date range:', finalDf['date'].min(), 'to', finalDf['date'].max())




In [ ]:
def _prepare_trade_frame(trades, return_col='ret_1d_next', date_col='date'):
    if isinstance(trades, pd.DataFrame):
        out = trades.copy()
        if return_col not in out.columns:
            raise KeyError(f"'{return_col}' column is required for metric calculation.")
        out = out.rename(columns={return_col: 'ret'})
        out['date'] = pd.to_datetime(out[date_col]) if date_col in out.columns else pd.NaT
        return out[['date', 'ret']].dropna(subset=['ret'])

    out = pd.DataFrame({'ret': pd.Series(trades).dropna()})
    out['date'] = pd.NaT
    return out[['date', 'ret']]


def _max_drawdown(return_series):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    equity = returns.cumsum()
    return float((equity - equity.cummax()).min() * 100)


def compute_metrics(trades, min_trades=MIN_TRADES_POOLED, calendar_dates=None):
    trade_df = _prepare_trade_frame(trades)
    if trade_df['date'].notna().any():
        trade_df = trade_df.sort_values('date').reset_index(drop=True)

    rets = trade_df['ret'].dropna()
    n = len(rets)

    if calendar_dates is not None:
        eval_days = len(pd.to_datetime(pd.Series(calendar_dates).dropna().unique()))
    elif trade_df['date'].notna().any():
        eval_days = len(pd.to_datetime(trade_df['date'].dropna().unique()))
    else:
        eval_days = np.nan

    annual_trades = n / (eval_days / TRADING_DAYS_PER_YEAR) if pd.notna(eval_days) and eval_days > 0 else np.nan

    if n == 0:
        return {
            'Trade Count': 0,
            'Avg Return': np.nan,
            'Win Rate': np.nan,
            'Sharpe': np.nan,
            'Sharpe_annualized': np.nan,
            'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
            'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
            'Max Drawdown': np.nan,
            'Note': ''
        }

    avg_ret = rets.mean() * 100
    win_rate = (rets > 0).mean() * 100
    mdd = _max_drawdown(rets)

    if n < min_trades:
        sharpe = np.nan
        sharpe_ann = np.nan
        note = f'N={n}<{min_trades} (소표본 경고: Sharpe 신뢰 불가)'
    else:
        std = rets.std()
        if std <= SHARPE_EPSILON:
            sharpe = np.nan
            sharpe_ann = np.nan
            note = '표준편차≈0 (Sharpe 신뢰 불가)'
        else:
            sharpe = float(rets.mean() / std)
            sharpe_ann = float(sharpe * np.sqrt(annual_trades)) if pd.notna(annual_trades) else np.nan
            note = ''

    return {
        'Trade Count': int(n),
        'Avg Return': float(avg_ret),
        'Win Rate': float(win_rate),
        'Sharpe': sharpe,
        'Sharpe_annualized': sharpe_ann,
        'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
        'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
        'Max Drawdown': float(mdd),
        'Note': note
    }


def apply_transaction_cost(trades, cost_bps=0):
    out = trades.copy()
    out['ret_1d_next'] = out['ret_1d_next'] - (cost_bps / 10000.0)
    return out



In [ ]:
q_list = [0.2, 0.3, 0.4, 0.5]
cost_bps_list = [0, 5, 10]
c_index_cols = finalDf.filter(like='C_', axis=1)

if c_index_cols.empty:
    raise ValueError('No C-index columns found in finalDf.')

print('C-index columns:', list(c_index_cols.columns))



In [ ]:
def evaluate_no_filter_with_cost(df, cost_bps=0, min_trades=MIN_TRADES_POOLED):
    df = df.copy().sort_values('date')
    trades = df.loc[df['y_hat'] == 1, ['date', 'ret_1d_next']].copy()
    net_trades = apply_transaction_cost(trades, cost_bps=cost_bps)
    metrics = compute_metrics(net_trades, min_trades=min_trades, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = 'No Filter'
    metrics['Quantile'] = np.nan
    metrics['Cost (bp)'] = cost_bps
    return metrics


def evaluate_filter_with_cost(df, c_col, quantile=0.3, cost_bps=0, min_trades=MIN_TRADES_POOLED):
    df = df.copy().sort_values('date')
    threshold = df[c_col].quantile(quantile)
    is_trade = (df['y_hat'] == 1) & (df[c_col] >= threshold)
    trades = df.loc[is_trade, ['date', 'ret_1d_next']].copy()
    net_trades = apply_transaction_cost(trades, cost_bps=cost_bps)
    metrics = compute_metrics(net_trades, min_trades=min_trades, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = c_col
    metrics['Quantile'] = quantile
    metrics['Threshold Value'] = threshold
    metrics['Cost (bp)'] = cost_bps
    return metrics


results = []

for cost_bps in cost_bps_list:
    for model_name, sub in finalDf.groupby('model'):
        m = evaluate_no_filter_with_cost(sub, cost_bps=cost_bps)
        m['Model'] = model_name
        results.append(m)

    for model_name, sub in finalDf.groupby('model'):
        for col in c_index_cols.columns:
            for q in q_list:
                m = evaluate_filter_with_cost(sub, col, quantile=q, cost_bps=cost_bps)
                m['Model'] = model_name
                results.append(m)

summaryTableCost = pd.DataFrame(results)

baseline_count_map = (
    summaryTableCost[summaryTableCost['C-Index Type'] == 'No Filter']
    .set_index(['Cost (bp)', 'Model'])['Trade Count']
    .to_dict()
)
summaryTableCost['Baseline Trade Count'] = [
    baseline_count_map[(cost, model)]
    for cost, model in zip(summaryTableCost['Cost (bp)'], summaryTableCost['Model'])
]
summaryTableCost['Delta Trade Count'] = summaryTableCost['Trade Count'] - summaryTableCost['Baseline Trade Count']

valid_pool_cost = summaryTableCost[
    (summaryTableCost['C-Index Type'] != 'No Filter') &
    (summaryTableCost['Trade Count'] >= MIN_TRADES_POOLED) &
    (summaryTableCost['Delta Trade Count'] < 0) &
    np.isfinite(summaryTableCost['Sharpe'])
]

best_rows_cost = (
    valid_pool_cost
    .sort_values(['Cost (bp)', 'Model', 'Sharpe'], ascending=[True, True, False])
    .groupby(['Cost (bp)', 'Model'])
    .first()
    .reset_index()
)
best_rows_cost = best_rows_cost.drop(columns=['Baseline Trade Count', 'Delta Trade Count'])

display(summaryTableCost)
display(best_rows_cost)



In [ ]:
baseline_cost = summaryTableCost[summaryTableCost['C-Index Type'] == 'No Filter'][
    ['Cost (bp)', 'Model', 'Trade Count', 'Avg Return', 'Win Rate', 'Sharpe',
     'Sharpe_annualized', 'Annual Trades', 'Max Drawdown']
].rename(columns={
    'Trade Count': 'Baseline Trade Count',
    'Avg Return': 'Baseline Avg Return',
    'Win Rate': 'Baseline Win Rate',
    'Sharpe': 'Baseline Sharpe',
    'Sharpe_annualized': 'Baseline Sharpe Annualized',
    'Annual Trades': 'Baseline Annual Trades',
    'Max Drawdown': 'Baseline Max Drawdown'
})

costImprovementTable = pd.merge(best_rows_cost, baseline_cost, on=['Cost (bp)', 'Model'], how='left')
costImprovementTable['Delta Trade Count'] = costImprovementTable['Trade Count'] - costImprovementTable['Baseline Trade Count']
costImprovementTable['Delta Avg Return'] = costImprovementTable['Avg Return'] - costImprovementTable['Baseline Avg Return']
costImprovementTable['Delta Win Rate'] = costImprovementTable['Win Rate'] - costImprovementTable['Baseline Win Rate']
costImprovementTable['Delta Sharpe'] = costImprovementTable['Sharpe'] - costImprovementTable['Baseline Sharpe']
costImprovementTable['Delta Sharpe Annualized'] = costImprovementTable['Sharpe_annualized'] - costImprovementTable['Baseline Sharpe Annualized']
costImprovementTable['Delta MDD'] = costImprovementTable['Max Drawdown'] - costImprovementTable['Baseline Max Drawdown']

costImprovementTable = costImprovementTable[[
    'Cost (bp)', 'Model', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Baseline Avg Return', 'Avg Return', 'Delta Avg Return',
    'Baseline Win Rate', 'Win Rate', 'Delta Win Rate',
    'Baseline Sharpe', 'Sharpe', 'Delta Sharpe',
    'Baseline Sharpe Annualized', 'Sharpe_annualized', 'Delta Sharpe Annualized',
    'Baseline Max Drawdown', 'Max Drawdown', 'Delta MDD',
    'Annual Trades', 'Eval Days', 'Note'
]].sort_values(['Cost (bp)', 'Model']).reset_index(drop=True)

display(costImprovementTable.round(4))



In [ ]:
def save_df_pretty(df, filename, dpi=450, title=None, include_index=False):
    df_show = df.copy()
    if include_index:
        df_show = df_show.reset_index()
    nrows, ncols = df_show.shape
    fig_w = max(10, ncols * 1.8)
    fig_h = max(2.5, (nrows + 1) * 0.52)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, pad=14)
    tbl = ax.table(cellText=df_show.values, colLabels=df_show.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.08, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.1)
        if r == 0:
            cell.set_text_props(weight='bold')
            cell.set_height(cell.get_height() * 1.12)
    plt.tight_layout(pad=1.5)
    fig.savefig(OUTPUT_DIR / filename, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)

save_df_pretty(summaryTableCost, 'cost_summary_performance_table.png', title='Cost-aware Summary Performance Table')
save_df_pretty(best_rows_cost, 'cost_best_scenarios_by_model.png', title='Best Scenarios by Model and Cost')
save_df_pretty(costImprovementTable.round(4), 'cost_improvement_table.png', title='Cost-aware Improvement over No Filter')

summaryTableCost.to_csv(TABLES_DIR / 'cost_summary_performance.csv', index=False)
best_rows_cost.to_csv(TABLES_DIR / 'cost_best_active_scenarios.csv', index=False)
costImprovementTable.to_csv(TABLES_DIR / 'cost_improvement.csv', index=False)

# Replace the main pooled tables with the corrected 0bp active-filter selection.
mainSummaryTable = summaryTableCost[summaryTableCost['Cost (bp)'] == 0].drop(columns=['Cost (bp)']).copy()
mainBestActive = best_rows_cost[best_rows_cost['Cost (bp)'] == 0].drop(columns=['Cost (bp)']).copy()
mainSummaryTable.to_csv(TABLES_DIR / 'summary_performance.csv', index=False)
mainBestActive.to_csv(TABLES_DIR / 'best_scenarios_by_model.csv', index=False)
save_df_pretty(mainBestActive, 'best_scenarios_by_model.png', title='Best Active Scenarios by Model (N >= 15, Trade Count Reduced)')

print(f'saved cost-aware tables to {OUTPUT_DIR}')



## Trade-quality statistical test

이 섹션은 위에서 만든 거래비용 반영 결과표(`summaryTableCost`, `best_rows_cost`, `costImprovementTable`) 아래에 이어서 실행한다.

검정 질문은 row-level 평균 수익률이 아니라, 교수님 피드백의 핵심인 다음 문제에 맞춘다.

> 6개 C-index 변형과 4개 q 후보를 탐색한 뒤 best를 고른 효과를 보정해도, active C-index filter가 No Filter 대비 per-trade 성과를 개선하는가?

따라서 main selection metric은 `Delta Sharpe`로 두고, active filter 후보만 best 선택에 포함한다.

- `Trade Count >= 15`: 소표본 Sharpe 폭발 방지
- `Delta Trade Count < 0`: 실제로 거래를 줄인 active filter만 포함
- `Delta Sharpe`: 교수님이 권고한 per-trade Sharpe 중심 지표
- `Delta Avg Return`, `Delta MDD`, `Delta Win Rate`: 보조 trade-quality 지표

Block bootstrap은 STEP=2 rolling window overlap을 고려해 20 trading-day block 단위로 신뢰구간을 계산한다. Block permutation + max-stat correction은 24개 후보 중 best를 사후 선택한 multiple testing 문제를 보정한다.

In [ ]:
# Trade-quality statistical test settings

SEED = 42
BLOCK_LENGTH_DAYS = 20
N_BOOTSTRAP = 2000
N_PERMUTATIONS = 2000
PROGRESS_EVERY = 100
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_DIR = CACHE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Main metric for best-candidate selection and max-stat correction.
SELECTION_METRIC = 'Delta Sharpe'

# Active filter conditions.
ACTIVE_MIN_TRADES = MIN_TRADES_POOLED

np.random.seed(SEED)

print('SELECTION_METRIC:', SELECTION_METRIC)
print('BLOCK_LENGTH_DAYS:', BLOCK_LENGTH_DAYS)
print('N_BOOTSTRAP:', N_BOOTSTRAP)
print('N_PERMUTATIONS:', N_PERMUTATIONS)
print('PROGRESS_EVERY:', PROGRESS_EVERY)
print('RESUME_FROM_CHECKPOINT:', RESUME_FROM_CHECKPOINT)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('ACTIVE_MIN_TRADES:', ACTIVE_MIN_TRADES)

### Candidate grid 재평가

위 거래비용 표는 이미 계산되어 있지만, 통계검정에서는 후보별 baseline 대비 개선폭과 active/no-op 여부가 필요하다.

여기서는 model-cost pair마다 24개 후보(`6 C-index x 4 q`)를 다시 정리하고, 다음 조건을 만족하는 후보만 best 선택 대상으로 둔다.

```text
Trade Count >= 15
Delta Trade Count < 0
Sharpe is finite
```

즉 MLP의 일부 `C_tau_full`처럼 거래 수를 전혀 줄이지 않는 no-op 후보는 main best 후보에서 제외된다.

In [ ]:
def _with_baseline_deltas(candidate_df, baseline_row):
    out = candidate_df.copy()
    base = baseline_row.copy()

    rename_map = {
        'Trade Count': 'Baseline Trade Count',
        'Avg Return': 'Baseline Avg Return',
        'Win Rate': 'Baseline Win Rate',
        'Sharpe': 'Baseline Sharpe',
        'Sharpe_annualized': 'Baseline Sharpe Annualized',
        'Max Drawdown': 'Baseline Max Drawdown'
    }

    for src, dst in rename_map.items():
        out[dst] = base[src]

    out['Delta Trade Count'] = out['Trade Count'] - out['Baseline Trade Count']
    out['Delta Avg Return'] = out['Avg Return'] - out['Baseline Avg Return']
    out['Delta Win Rate'] = out['Win Rate'] - out['Baseline Win Rate']
    out['Delta Sharpe'] = out['Sharpe'] - out['Baseline Sharpe']
    out['Delta Sharpe Annualized'] = out['Sharpe_annualized'] - out['Baseline Sharpe Annualized']
    out['Delta MDD'] = out['Max Drawdown'] - out['Baseline Max Drawdown']

    out['Is Active Filter'] = (
        (out['Trade Count'] >= ACTIVE_MIN_TRADES) &
        (out['Delta Trade Count'] < 0) &
        np.isfinite(out['Sharpe']) &
        np.isfinite(out['Baseline Sharpe'])
    )
    out['Is Positive Sharpe'] = out['Delta Sharpe'] > 0
    return out


def evaluate_trade_quality_grid(df, cost_bps=0, c_cols=None, q_values=None):
    c_cols = list(c_cols or c_index_cols.columns)
    q_values = list(q_values or q_list)

    baseline = evaluate_no_filter_with_cost(df, cost_bps=cost_bps)

    rows = []
    for c_col in c_cols:
        for q in q_values:
            m = evaluate_filter_with_cost(df, c_col, quantile=q, cost_bps=cost_bps)
            rows.append(m)

    grid = pd.DataFrame(rows)
    grid = _with_baseline_deltas(grid, pd.Series(baseline))
    return grid


def build_trade_quality_candidate_grid(final_df):
    rows = []
    for cost_bps in cost_bps_list:
        for model_name, sub in final_df.groupby('model'):
            grid = evaluate_trade_quality_grid(sub, cost_bps=cost_bps)
            grid['Model'] = model_name
            grid['Cost (bp)'] = cost_bps
            rows.append(grid)
    return pd.concat(rows, ignore_index=True)


tradeQualityCandidateGrid = build_trade_quality_candidate_grid(finalDf)
activeTradeQualityGrid = tradeQualityCandidateGrid[tradeQualityCandidateGrid['Is Active Filter']].copy()

tradeQualityGridSummary = (
    tradeQualityCandidateGrid
    .groupby(['Model', 'Cost (bp)'])
    .agg(
        Candidate_Count=('C-Index Type', 'count'),
        Active_Candidates=('Is Active Filter', 'sum'),
        Positive_Active_Candidates=('Is Positive Sharpe', lambda x: int((x & tradeQualityCandidateGrid.loc[x.index, 'Is Active Filter']).sum())),
        Mean_Delta_Sharpe=('Delta Sharpe', 'mean'),
        Median_Delta_Sharpe=('Delta Sharpe', 'median'),
        Best_Delta_Sharpe=('Delta Sharpe', 'max'),
        Worst_Delta_Sharpe=('Delta Sharpe', 'min'),
        Mean_Delta_Avg_Return=('Delta Avg Return', 'mean'),
        Best_Delta_Avg_Return=('Delta Avg Return', 'max')
    )
    .reset_index()
)

bestActiveTradeQuality = (
    activeTradeQualityGrid
    .sort_values(['Model', 'Cost (bp)', SELECTION_METRIC], ascending=[True, True, False])
    .groupby(['Model', 'Cost (bp)'])
    .first()
    .reset_index()
)

show_cols = [
    'Model', 'Cost (bp)', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Baseline Avg Return', 'Avg Return', 'Delta Avg Return',
    'Baseline Win Rate', 'Win Rate', 'Delta Win Rate',
    'Baseline Sharpe', 'Sharpe', 'Delta Sharpe',
    'Baseline Max Drawdown', 'Max Drawdown', 'Delta MDD',
    'Threshold Value'
]

print('candidate grid:', tradeQualityCandidateGrid.shape)
print('active candidate grid:', activeTradeQualityGrid.shape)
display(tradeQualityGridSummary.round(6))
display(bestActiveTradeQuality[show_cols].round(6))

### Block bootstrap: best active filter의 신뢰구간

각 model-cost pair에서 `Delta Sharpe` 기준으로 선택된 best active candidate를 고정하고, 날짜를 20 trading-day block 단위로 재표본추출한다.

목적은 선택된 후보의 개선폭이 표본 변동에 얼마나 민감한지 보는 것이다. 여기서는 `Delta Sharpe`, `Delta Avg Return`, `Delta Win Rate`, `Delta MDD`의 95% CI를 계산한다.

In [ ]:
def sample_dates_by_blocks(unique_dates, rng, block_length=BLOCK_LENGTH_DAYS):
    dates = pd.to_datetime(pd.Series(unique_dates).dropna().sort_values().unique())
    n = len(dates)
    sampled = []
    while len(sampled) < n:
        start = int(rng.integers(0, n))
        for k in range(block_length):
            sampled.append(dates[(start + k) % n])
            if len(sampled) >= n:
                break
    return sampled[:n]


def bootstrap_sample_by_date_blocks(df, rng, block_length=BLOCK_LENGTH_DAYS):
    sampled_dates = sample_dates_by_blocks(df['date'].unique(), rng, block_length=block_length)
    if not sampled_dates:
        return df.iloc[0:0].copy()
    draw_order = pd.DataFrame({
        'date': pd.to_datetime(sampled_dates),
        '_bootstrap_order': np.arange(len(sampled_dates))
    })
    out = draw_order.merge(df, on='date', how='left', sort=False)
    return out.sort_values(['_bootstrap_order', 'asset']).drop(columns=['_bootstrap_order'])


def evaluate_one_policy(df, c_col, q, cost_bps):
    baseline = evaluate_no_filter_with_cost(df, cost_bps=cost_bps)
    candidate = evaluate_filter_with_cost(df, c_col, quantile=q, cost_bps=cost_bps)
    return _with_baseline_deltas(pd.DataFrame([candidate]), pd.Series(baseline)).iloc[0]


def _safe_checkpoint_token(value):
    return ''.join(ch if str(ch).isalnum() else '_' for ch in str(value))


def _run_signature(df):
    max_date = pd.to_datetime(df['date']).max().strftime('%Y%m%d')
    return f'n{len(df)}_d{max_date}'


def bootstrap_best_active_trade_quality(final_df, best_table, n_bootstrap=N_BOOTSTRAP, block_length=BLOCK_LENGTH_DAYS, seed=SEED):
    pair_frames = []
    model_offsets = {'gbm': 100000, 'mlp': 200000, 'rf': 300000}

    for _, best in best_table.iterrows():
        model_name = best['Model']
        cost_bps = int(best['Cost (bp)'])
        c_col = best['C-Index Type']
        q = float(best['Quantile'])
        sub = final_df[final_df['model'] == model_name].copy()
        signature = _run_signature(sub)
        checkpoint = CHECKPOINT_DIR / (
            f'bootstrap_{_safe_checkpoint_token(model_name)}_{cost_bps}_'
            f'{_safe_checkpoint_token(c_col)}_{q}_{signature}.pkl'
        )

        if RESUME_FROM_CHECKPOINT and checkpoint.exists():
            pair_df = pd.read_pickle(checkpoint)
            pair_df = pair_df[pair_df['Bootstrap Iteration'] < n_bootstrap].copy()
        else:
            pair_df = pd.DataFrame()

        completed = set(pair_df.get('Bootstrap Iteration', pd.Series(dtype=int)).astype(int))
        pair_rows = pair_df.to_dict('records')
        print(f'[bootstrap start] model={model_name} cost={cost_bps} resumed={len(completed)}/{n_bootstrap}')

        for b in range(n_bootstrap):
            if b in completed:
                continue
            # Reuse the same resampled dates across cost scenarios for a paired comparison.
            iteration_seed = seed + model_offsets.get(str(model_name), 400000) + b
            rng = np.random.default_rng(iteration_seed)
            boot = bootstrap_sample_by_date_blocks(sub, rng, block_length=block_length)
            m = evaluate_one_policy(boot, c_col=c_col, q=q, cost_bps=cost_bps)
            pair_rows.append({
                'Model': model_name,
                'Cost (bp)': cost_bps,
                'C-Index Type': c_col,
                'Quantile': q,
                'Bootstrap Iteration': b,
                'Delta Sharpe': m['Delta Sharpe'],
                'Delta Avg Return': m['Delta Avg Return'],
                'Delta Win Rate': m['Delta Win Rate'],
                'Delta MDD': m['Delta MDD'],
                'Trade Count': m['Trade Count'],
                'Delta Trade Count': m['Delta Trade Count']
            })

            if (b + 1) % PROGRESS_EVERY == 0 or b + 1 == n_bootstrap:
                pair_df = pd.DataFrame(pair_rows).sort_values('Bootstrap Iteration').reset_index(drop=True)
                pair_df.to_pickle(checkpoint)
                print(f'[bootstrap progress] model={model_name} cost={cost_bps} {len(pair_df)}/{n_bootstrap}')

        pair_df = pd.DataFrame(pair_rows).sort_values('Bootstrap Iteration').reset_index(drop=True)
        pair_df.to_pickle(checkpoint)
        pair_frames.append(pair_df)
        print(f'[bootstrap done] model={model_name} cost={cost_bps} checkpoint={checkpoint.name}')

    return pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()


def summarize_trade_quality_bootstrap(boot_df):
    rows = []
    metrics = ['Delta Sharpe', 'Delta Avg Return', 'Delta Win Rate', 'Delta MDD']
    for keys, grp in boot_df.groupby(['Model', 'Cost (bp)', 'C-Index Type', 'Quantile']):
        row = dict(zip(['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'], keys))
        for metric in metrics:
            vals = grp[metric].replace([np.inf, -np.inf], np.nan).dropna()
            row[f'{metric} CI Low'] = float(vals.quantile(0.025)) if len(vals) else np.nan
            row[f'{metric} CI High'] = float(vals.quantile(0.975)) if len(vals) else np.nan
            row[f'{metric} Boot Mean'] = float(vals.mean()) if len(vals) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


tradeQualityBootstrapDistribution = bootstrap_best_active_trade_quality(finalDf, bestActiveTradeQuality)
tradeQualityBootstrapSummary = summarize_trade_quality_bootstrap(tradeQualityBootstrapDistribution)

display(tradeQualityBootstrapSummary.round(6))

### Block permutation + max-stat correction

귀무가설은 다음과 같다.

```text
C-index 값은 미래 거래 성과와 무관하다.
```

검정 절차는 다음과 같다.

1. 날짜 block 단위로 C-index columns만 permutation한다.
2. `ret_1d_next`, `y_hat`, `date`, `asset` 구조는 유지한다.
3. 각 permutation에서 24개 후보를 모두 평가한다.
4. active filter 조건을 동일하게 적용한다.
5. 그 permutation 안에서 최대 `Delta Sharpe`를 저장한다.
6. 실제 observed best `Delta Sharpe`가 이 null max distribution에서 얼마나 극단적인지 adjusted p-value로 계산한다.

즉 단순히 best filter와 No Filter를 비교하는 것이 아니라, 24개 후보를 뒤져서 best를 고르면 우연히 이 정도 개선이 나올 수 있는가를 검정한다.

In [ ]:
def make_date_blocks_for_permutation(df, block_length=BLOCK_LENGTH_DAYS):
    df_sorted = df.sort_values(['date', 'asset']).reset_index(drop=True).copy()
    unique_dates = pd.to_datetime(pd.Series(df_sorted['date'].dropna().unique())).sort_values().to_numpy()
    blocks = []
    for i in range(0, len(unique_dates), block_length):
        block_dates = unique_dates[i:i + block_length]
        positions = np.flatnonzero(df_sorted['date'].isin(block_dates).to_numpy())
        blocks.append(positions)
    return df_sorted, blocks


def permute_cindex_values_by_blocks(original_values, blocks, rng):
    if len(blocks) <= 1:
        return original_values.copy()
    target_positions = np.concatenate(blocks)
    source_positions = np.concatenate([blocks[i] for i in rng.permutation(len(blocks))])
    out = original_values.copy()
    out[target_positions, :] = original_values[source_positions, :]
    return out


def _fast_sharpe(return_values, cost_bps):
    values = np.asarray(return_values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < ACTIVE_MIN_TRADES:
        return np.nan
    std = values.std(ddof=1)
    if not np.isfinite(std) or std <= SHARPE_EPSILON:
        return np.nan
    return float((values.mean() - cost_bps / 10000.0) / std)


def fast_permutation_delta_grid(df_sorted, permuted_values, c_cols, cost_bps):
    base_mask = df_sorted['y_hat'].to_numpy() == 1
    returns = df_sorted['ret_1d_next'].to_numpy(dtype=float)
    baseline_count = int(base_mask.sum())
    baseline_sharpe = _fast_sharpe(returns[base_mask], cost_bps)
    rows = []

    for col_idx, c_col in enumerate(c_cols):
        c_values = permuted_values[:, col_idx]
        for q in q_list:
            threshold = float(np.nanquantile(c_values, q))
            trade_mask = base_mask & (c_values >= threshold)
            trade_count = int(trade_mask.sum())
            sharpe = _fast_sharpe(returns[trade_mask], cost_bps)
            is_active = (
                trade_count >= ACTIVE_MIN_TRADES and
                trade_count < baseline_count and
                np.isfinite(sharpe) and
                np.isfinite(baseline_sharpe)
            )
            rows.append({
                'C-Index Type': c_col,
                'Quantile': q,
                'Trade Count': trade_count,
                'Delta Trade Count': trade_count - baseline_count,
                'Delta Sharpe': sharpe - baseline_sharpe if is_active else np.nan,
                'Is Active Filter': is_active
            })
    return pd.DataFrame(rows)


def _pvalue_from_null(observed, null_values):
    null_values = pd.Series(null_values).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan
    return float((1 + np.sum(null_values >= observed)) / (len(null_values) + 1))


def permutation_max_stat_trade_quality(final_df, best_table, n_permutations=N_PERMUTATIONS, block_length=BLOCK_LENGTH_DAYS, seed=SEED):
    p_rows = []
    pair_frames = []
    model_offsets = {'gbm': 1000000, 'mlp': 2000000, 'rf': 3000000}

    best_lookup = {
        (row['Model'], int(row['Cost (bp)'])): row
        for _, row in best_table.iterrows()
    }

    for model_name, sub in final_df.groupby('model'):
        c_cols = list(c_index_cols.columns)
        sub_sorted, blocks = make_date_blocks_for_permutation(sub, block_length=block_length)
        original_values = sub_sorted[c_cols].to_numpy(dtype=float)
        signature = _run_signature(sub_sorted)

        for cost_bps in cost_bps_list:
            if (model_name, int(cost_bps)) not in best_lookup:
                print(f'[skip] no active observed candidate: model={model_name} cost={cost_bps}')
                continue

            observed_best = best_lookup[(model_name, int(cost_bps))]
            observed_metric = float(observed_best[SELECTION_METRIC])
            selected_c = observed_best['C-Index Type']
            selected_q = float(observed_best['Quantile'])
            checkpoint = CHECKPOINT_DIR / (
                f'permutation_{_safe_checkpoint_token(model_name)}_{cost_bps}_{signature}.pkl'
            )

            if RESUME_FROM_CHECKPOINT and checkpoint.exists():
                pair_df = pd.read_pickle(checkpoint)
                pair_df = pair_df[pair_df['Permutation'] < n_permutations].copy()
            else:
                pair_df = pd.DataFrame()

            completed = set(pair_df.get('Permutation', pd.Series(dtype=int)).astype(int))
            pair_rows = pair_df.to_dict('records')
            print(f'[permutation start] model={model_name} cost={cost_bps} resumed={len(completed)}/{n_permutations}')

            for p in range(n_permutations):
                if p in completed:
                    continue
                iteration_seed = seed + model_offsets.get(str(model_name), 4000000) + p
                rng = np.random.default_rng(iteration_seed)
                permuted_values = permute_cindex_values_by_blocks(original_values, blocks, rng)
                perm_grid = fast_permutation_delta_grid(sub_sorted, permuted_values, c_cols, cost_bps)
                perm_active = perm_grid[perm_grid['Is Active Filter']].copy()

                if perm_active.empty:
                    selected_val = np.nan
                    max_val = np.nan
                else:
                    max_val = float(perm_active[SELECTION_METRIC].max())
                    selected_match = perm_active[
                        (perm_active['C-Index Type'] == selected_c) &
                        (np.isclose(perm_active['Quantile'].astype(float), selected_q))
                    ]
                    selected_val = float(selected_match[SELECTION_METRIC].iloc[0]) if len(selected_match) else np.nan

                pair_rows.append({
                    'Model': model_name,
                    'Cost (bp)': cost_bps,
                    'Permutation': p,
                    'Selected Candidate Null': selected_val,
                    'Max Active Null': max_val
                })

                if (p + 1) % PROGRESS_EVERY == 0 or p + 1 == n_permutations:
                    pair_df = pd.DataFrame(pair_rows).sort_values('Permutation').reset_index(drop=True)
                    pair_df.to_pickle(checkpoint)
                    print(f'[permutation progress] model={model_name} cost={cost_bps} {len(pair_df)}/{n_permutations}')

            pair_df = pd.DataFrame(pair_rows).sort_values('Permutation').reset_index(drop=True)
            pair_df.to_pickle(checkpoint)
            pair_frames.append(pair_df)
            selected_null = pair_df['Selected Candidate Null']
            max_null = pair_df['Max Active Null']

            raw_p = _pvalue_from_null(observed_metric, selected_null)
            adjusted_p = _pvalue_from_null(observed_metric, max_null)

            p_rows.append({
                'Model': model_name,
                'Cost (bp)': cost_bps,
                'C-Index Type': selected_c,
                'Quantile': selected_q,
                'Observed Best Metric': observed_metric,
                'Raw p': raw_p,
                'Adjusted p (max-stat)': adjusted_p,
                'Null Selected Mean': float(pd.Series(selected_null).mean()),
                'Null Selected 95%': float(pd.Series(selected_null).quantile(0.95)),
                'Null Max Mean': float(pd.Series(max_null).mean()),
                'Null Max 95%': float(pd.Series(max_null).quantile(0.95)),
                'N Permutations': n_permutations,
                'Block Length Days': block_length
            })

            print(f'[permutation done] model={model_name} cost={cost_bps} raw_p={raw_p:.4f} adj_p={adjusted_p:.4f} checkpoint={checkpoint.name}')

    null_table = pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()
    return null_table, pd.DataFrame(p_rows)


tradeQualityPermutationNullTable, tradeQualityPValueTable = permutation_max_stat_trade_quality(finalDf, bestActiveTradeQuality)

display(tradeQualityPValueTable.round(6))

### Final trade-quality test table

이 최종표는 기존 거래비용 결과표와 통계검정 결과를 합친다.

해석 기준은 다음과 같다.

- `Adjusted p < 0.05`: 24개 후보 사후 선택 보정 후에도 유의
- `0.05 <= Adjusted p < 0.10`: 보정 후 약한/suggestive evidence
- `Raw p`만 낮고 `Adjusted p`가 높음: best 후보는 좋아 보이나 multiple testing correction 후 비유의
- 둘 다 높음: exploratory finding으로만 해석

In [ ]:
def _decision_label(p):
    if pd.isna(p):
        return 'no valid active candidate'
    if p < 0.05:
        return 'significant after max-stat correction'
    if p < 0.10:
        return 'suggestive after max-stat correction'
    return 'not significant after correction'


tradeQualityFinalTestTable = (
    bestActiveTradeQuality[show_cols]
    .merge(
        tradeQualityBootstrapSummary,
        on=['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'],
        how='left'
    )
    .merge(
        tradeQualityPValueTable,
        on=['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'],
        how='left'
    )
)

tradeQualityFinalTestTable['Decision'] = tradeQualityFinalTestTable['Adjusted p (max-stat)'].apply(_decision_label)

final_cols = [
    'Model', 'Cost (bp)', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Delta Avg Return', 'Delta Avg Return CI Low', 'Delta Avg Return CI High',
    'Delta Sharpe', 'Delta Sharpe CI Low', 'Delta Sharpe CI High',
    'Delta Win Rate', 'Delta Win Rate CI Low', 'Delta Win Rate CI High',
    'Delta MDD', 'Delta MDD CI Low', 'Delta MDD CI High',
    'Raw p', 'Adjusted p (max-stat)', 'Decision'
]

tradeQualityFinalTestTable = tradeQualityFinalTestTable[final_cols].sort_values(['Cost (bp)', 'Model']).reset_index(drop=True)

display(tradeQualityFinalTestTable.round(6))

In [ ]:
# Save trade-quality statistical test artifacts

ARTIFACT_DIR = CACHE_DIR

objects_to_save = {
    'trade_quality_candidate_grid': tradeQualityCandidateGrid,
    'trade_quality_grid_summary': tradeQualityGridSummary,
    'trade_quality_best_active_table': bestActiveTradeQuality,
    'trade_quality_bootstrap_distribution': tradeQualityBootstrapDistribution,
    'trade_quality_bootstrap_summary': tradeQualityBootstrapSummary,
    'trade_quality_permutation_null_table': tradeQualityPermutationNullTable,
    'trade_quality_pvalue_table': tradeQualityPValueTable,
    'trade_quality_final_test_table': tradeQualityFinalTestTable,
}

for name, obj in objects_to_save.items():
    pkl_path = ARTIFACT_DIR / f'{name}.pkl'
    csv_path = TABLES_DIR / f'{name}.csv'
    obj.to_pickle(pkl_path)
    obj.to_csv(csv_path, index=False)
    print(f'[saved] {pkl_path}')
    print(f'[saved] {csv_path}')

save_df_pretty(tradeQualityGridSummary.round(6), 'trade_quality_grid_summary.png', title='Trade-quality Active Candidate Summary')
save_df_pretty(bestActiveTradeQuality[show_cols].round(6), 'trade_quality_best_active_candidates.png', title='Best Active C-index Filters by Delta Sharpe')
save_df_pretty(tradeQualityFinalTestTable.round(6), 'trade_quality_final_test_table.png', title='Trade-quality Bootstrap and Max-stat Test')

print(f'saved trade-quality statistical test tables to {ARTIFACT_DIR} and {OUTPUT_DIR}')

### Interpretation note

이 검정의 결론 문장은 `tradeQualityFinalTestTable`의 `Adjusted p (max-stat)`를 기준으로 작성한다.

- 보정 후 유의하지 않으면: “C-index filter는 거래비용 반영 후 일부 per-trade Sharpe/MDD 개선 경향을 보였으나, 24개 후보 탐색에 따른 max-stat correction 이후 통계적으로 유의한 개선은 확인되지 않았다.”
- 보정 후 유의하면: “active filter 후보와 multiple testing correction을 고려한 뒤에도 일부 model-cost pair에서 C-index filter의 trade-quality improvement가 통계적으로 지지되었다.”

이 섹션은 기존의 row-level paired Delta Mean Return 검정이 아니라, 본 연구의 reliability filter 프레이밍에 맞춘 per-trade quality 검정이다.